Exact-Context Vote Consistency Analysis

"""
Case A — Vote Consistency Under Strictly Identical Conditions
==============================================================

This analysis evaluates the consistency of thermal comfort votes when occupants
are exposed to **exactly the same conditions**, meaning:

- identical environmental variables (`Tair`, `RH`, `Tout`, `vel`)
- identical personal variables (`clo`, `Met`, `Age`, `Sex`)
- identical contextual variables (`Season`, `Climate`, `Building_type`, `cooling type`)

Objective
---------

Determine to what extent two individuals experiencing *strictly identical*
conditions give the same or different thermal comfort votes.

Methodology
-----------

1. Select and filter all relevant numerical and categorical features.
2. Construct a unique key combining all contextual variables.
3. Group all rows that share the exact same key.
4. For each target variable:
   - compute the number of groups with identical votes,
   - compute the number of groups with differing votes,
   - estimate the agreement and disagreement rates.
5. Display and export JSON files containing the contradictory groups.

"""


In [1]:
import pandas as pd
from collections import Counter
import json
from pathlib import Path

OUTPUT_DIR = Path("Votes_analysis/analysis_results_exact_context")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===========================
# 1. Load dataset
# ===========================
df = pd.read_csv("../Data/ASHRAE_2022_Clean_api.csv")


# ===========================
# 2. Columns of interest
# ===========================
features = [
    'Tair', 'RH', 'clo', 'vel', 'Âge', 'Tout', 'Met', 'Sexe',
    'Season', 'Climate', 'Building_type', 'cooling type'
]

targets = ['thermal_sensation', 'TSV_3p', 'thermal_preference']
targets = [t for t in targets if t in df.columns]

data = df[features + targets].dropna()

print("Dataset size used:", len(data))

# ===========================
# 3. Group by exact context
# ===========================
grouped = data.groupby(features)

groups = [g for _, g in grouped if len(g) > 1]

print("Number of strictly identical groups:", len(groups))

# ===========================
# 4. Agreement/disagreement function
# ===========================
def analyze_target(target):
    same = 0
    diff = 0
    details = []

    for g in groups:
        votes = g[target].tolist()
        unique_votes = set(votes)

        if len(unique_votes) == 1:
            same += 1
        else:
            diff += 1
            details.append(unique_votes)

    print(f"\n=== Analysis for {target} ===")
    print("Total groups:", len(groups))
    print("→ Identical votes:", same)
    print("→ Different votes:", diff)
    print("→ Agreement rate:", round(same / len(groups), 3))
    print("→ Disagreement rate:", round(diff / len(groups), 3))

    return details

# ===========================
# 5. Run for each target
# ===========================
for t in targets:
    analyze_target(t)

# ===========================
# 6. Detailed per-group analysis
# ===========================
group_details_identical = {t: [] for t in targets}
group_details_different = {t: [] for t in targets}

for name, g in grouped:
    if len(g) < 2:
        continue

    context_dict = dict(zip(features, name))

    for target in targets:
        votes = g[target].tolist()
        unique_votes = set(votes)

        group_info = {
            "features": context_dict,
            "votes": votes,
            "unique_votes": list(unique_votes),
            "size": len(votes)
        }

        if len(unique_votes) == 1:
            group_details_identical[target].append(group_info)
        else:
            group_details_different[target].append(group_info)

# ======================================
# SAVE RESULTS FOR EACH TARGET IN JSON
# ======================================

summary_global = {}

for target in targets:

    out = OUTPUT_DIR / target
    out.mkdir(exist_ok=True)

    summary = {
        "target": target,
        "num_identical_groups": len(group_details_identical[target]),
        "num_different_groups": len(group_details_different[target]),
        "agreement_rate": round(len(group_details_identical[target]) / 
                                (len(group_details_identical[target]) + len(group_details_different[target])), 3)
        if (len(group_details_identical[target]) + len(group_details_different[target])) > 0 else None,
        "disagreement_rate": round(len(group_details_different[target]) / 
                                  (len(group_details_identical[target]) + len(group_details_different[target])), 3)
        if (len(group_details_identical[target]) + len(group_details_different[target])) > 0 else None
    }

    summary_global[target] = summary

    with open(out / "summary.json", "w") as f:
        json.dump(summary, f, indent=4)

    with open(out / "groups_identical.json", "w") as f:
        json.dump(group_details_identical[target], f, indent=4)

    with open(out / "groups_different.json", "w") as f:
        json.dump(group_details_different[target], f, indent=4)

# ======================================
# SAVE GLOBAL SUMMARY
# ======================================

with open(OUTPUT_DIR / "global_summary.json", "w") as f:
    json.dump(summary_global, f, indent=4)

print("\nJSON results saved in:", OUTPUT_DIR)

Dataset size used: 15224
Number of strictly identical groups: 22

=== Analysis for thermal_sensation ===
Total groups: 22
→ Identical votes: 14
→ Different votes: 8
→ Agreement rate: 0.636
→ Disagreement rate: 0.364

=== Analysis for thermal_preference ===
Total groups: 22
→ Identical votes: 17
→ Different votes: 5
→ Agreement rate: 0.773
→ Disagreement rate: 0.227

JSON results saved in: Votes_analysis/analysis_results_exact_context


Rounded Similar Contexts

"""
Case A.1 — Similar Context via 0.1 Rounding (Numerical Features Rounded)
========================================================================

This analysis evaluates the consistency of thermal comfort votes when occupants
experience **almost identical** conditions, defined as:

- identical categorical variables,
- numerical variables rounded to one decimal place (`round(x, 1)`).

Objective
---------

Assess whether a slight relaxation of the context (approximately ±0.05 tolerance)
increases the variability of votes compared to the strict identical-condition case.

Methodology
-----------

1. Round each numerical feature to one decimal place.
2. Build a composite key combining all rounded features.
3. Group all rows that share the same rounded-context key.
4. Compute agreement and disagreement rates for each target variable.
5. Extract the top 10 most contradictory groups (maximum vote spread).
6. Export JSON files containing summaries and contradictory groups.

"""


> ⚠️ **Exploratory variant — not published in the B&E manuscript.** This cell applies a coarse pre-rounding of numerical features before grouping to probe the sensitivity of the identical-cluster analysis to the grouping granularity. The published figures in §3.6 Case B come from Cell 2 (strict equality on raw values, 22 clusters) and from the canonical script `nn_disagreement.py`.

In [2]:
import pandas as pd
from collections import defaultdict
import json
from pathlib import Path

# ===========================
# 1. Load dataset
# ===========================
df = pd.read_csv("../Data/ASHRAE_2022_Clean_api.csv")

num_vars = ['Tair', 'RH', 'clo', 'vel', 'Tout', 'Met']
cat_vars = ['Âge', 'Sexe', 'Season', 'Climate', 'Building_type', 'cooling type']

targets = ['thermal_sensation', 'TSV_3p', 'thermal_preference']
targets = [t for t in targets if t in df.columns]

data = df[num_vars + cat_vars + targets].dropna().reset_index(drop=True)

print("Dataset size:", len(data))

OUTPUT_DIR = Path("Votes_analysis/analysis_results_rounded_context")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ===========================
# 2. Round numerical variables (1 decimal)
# ===========================
for var in num_vars:
    data[var + "_r"] = data[var].round(1)

group_cols = [v + "_r" for v in num_vars] + cat_vars

# ===========================
# 3. Create grouping key
# ===========================
data['group_key'] = data[group_cols].astype(str).agg("_".join, axis=1)

# ===========================
# 4. Build clusters
# ===========================
groups = data.groupby('group_key')
clusters = [grp.index.tolist() for _, grp in groups if len(grp) > 1]

print("Nombre de groupes similaires (arrondi 1 décimale) :", len(clusters))

def analyze_target_rounded(target):
    same = 0
    diff = 0

    for grp in clusters:
        votes = list(data.loc[grp, target])

        if len(set(votes)) == 1:
            same += 1
        else:
            diff += 1

    summary = {
        "target": target,
        "total_groups": len(clusters),
        "identical_vote_groups": same,
        "different_vote_groups": diff,
        "agreement_rate": round(same / len(clusters), 3),
        "disagreement_rate": round(diff / len(clusters), 3)
    }

    print(f"\n=== Analyse pour {target} ===")
    for k, v in summary.items():
        print(f"{k} : {v}")

    return summary


def get_top_k_contradictions(target, TOP_K=10):
    other_target = [t for t in targets if t != target][0]

    groups_info = []

    for grp in clusters:
        votes = list(data.loc[grp, target])
        if len(set(votes)) > 1:   
            spread = max(votes) - min(votes)
            groups_info.append({
                "indices": grp,
                "spread": spread,
                "size": len(grp),
                "votes": votes
            })

    groups_info = sorted(groups_info, key=lambda x: x["spread"], reverse=True)

    return groups_info[:TOP_K]

def save_results_for_target(target, summary, top_groups):
    out_dir = OUTPUT_DIR / target
    out_dir.mkdir(exist_ok=True)

    with open(out_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=4)

    detailed = []

    for info in top_groups:
        grp = info["indices"]

        detailed.append({
            "size": info["size"],
            "spread": info["spread"],
            "votes_target": info["votes"],
            "context_features": data.loc[grp, group_cols].drop_duplicates().to_dict(orient="records"),
            "votes_other_target": data.loc[grp, [t for t in targets if t != target]].to_dict(orient="records"),
        })

    with open(out_dir / "top_groups.json", "w") as f:
        json.dump(detailed, f, indent=4)

    print(f"\n→ Results saved to: {out_dir}")

global_summary = {}

for target in targets:
    print("\n" + "="*70)
    print(f"🔍 ANALYSE POUR TARGET : {target}")
    print("="*70)

    summary = analyze_target_rounded(target)
    global_summary[target] = summary

    top_groups = get_top_k_contradictions(target, TOP_K=10)

    print(f"\nTop {len(top_groups)} contradictory groups for {target}:")
    for i, grp in enumerate(top_groups):
        print(f"\n--- Groupe #{i+1} ---")
        print("Taille :", grp["size"])
        print("Spread :", grp["spread"])
        print("Votes :", grp["votes"])

        print("\nContexte :")
        print(data.loc[grp["indices"], group_cols].drop_duplicates())

    save_results_for_target(target, summary, top_groups)

with open(OUTPUT_DIR / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=4)

print("\n🔥 Analyse terminée pour toutes les targets.")


Dataset size: 15224
Nombre de groupes similaires (arrondi 1 décimale) : 92

🔍 ANALYSE POUR TARGET : thermal_sensation

=== Analyse pour thermal_sensation ===
target : thermal_sensation
total_groups : 92
identical_vote_groups : 40
different_vote_groups : 52
agreement_rate : 0.435
disagreement_rate : 0.565

Top 10 contradictory groups for thermal_sensation:

--- Groupe #1 ---
Taille : 2
Spread : 4
Votes : [-2, 2]

Contexte :
     Tair_r  RH_r  clo_r  vel_r  Tout_r  Met_r   Âge  Sexe  Season Climate  \
954    23.4  59.2    0.7    0.3    23.6    1.1  31.0  male  summer     Cfa   

    Building_type     cooling type  
954        office  air conditioned  

--- Groupe #2 ---
Taille : 2
Spread : 3
Votes : [-3, 0]

Contexte :
      Tair_r  RH_r  clo_r  vel_r  Tout_r  Met_r   Âge    Sexe  Season Climate  \
3527    21.0  36.7    0.8    0.1    15.8    1.1  18.0  female  summer     Cfb   

     Building_type          cooling type  
3527        office  naturally ventilated  

--- Groupe #3 ---
Taill

Round-to-Int Similar Context

"""
Case A.2 — Similar Context via Integer Rounding (±0.5 Margin)
==============================================================

In this analysis, numerical variables are rounded to the nearest integer,
which produces broader context groups:

- ±0.5°C for Tair and Tout  
- ±0.5 for Met, clo, vel  
- ±0.5 for RH (already a relatively wide margin)

Objective
---------

Examine how thermal comfort votes behave when occupants are placed in a
**coarse-grained context**, where small environmental differences are
intentionally ignored.

Methodology
-----------

1. Round each numerical feature using `round(x)`.
2. Construct a composite key combining all rounded features.
3. Group all rows sharing the same integer-rounded context.
4. Analyze the vote spread within each group.
5. Save the top 10 most contradictory groups.

"""


> ⚠️ **Exploratory variant — not published in the B&E manuscript.** This cell applies a coarse pre-rounding of numerical features before grouping to probe the sensitivity of the identical-cluster analysis to the grouping granularity. The published figures in §3.6 Case B come from Cell 2 (strict equality on raw values, 22 clusters) and from the canonical script `nn_disagreement.py`.

In [3]:
import pandas as pd
from collections import defaultdict
import json
from pathlib import Path

# ===========================
# 1. Load dataset
# ===========================
df = pd.read_csv("../Data/ASHRAE_2022_Clean_api.csv")

num_vars = ['Tair', 'RH', 'clo', 'vel', 'Tout', 'Met']
cat_vars = ['Âge', 'Sexe', 'Season', 'Climate', 'Building_type', 'cooling type']

targets = ['thermal_sensation', 'TSV_3p', 'thermal_preference']
targets = [t for t in targets if t in df.columns]

data = df[num_vars + cat_vars + targets].dropna().reset_index(drop=True)

print("Dataset size:", len(data))

OUTPUT_DIR = Path("Votes_analysis/analysis_results_rounded_int_context")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ===========================
# 2. Round numerical variables to integers
# ===========================
for var in num_vars:
    data[var + "_r"] = data[var].round().astype(int)

group_cols = [v + "_r" for v in num_vars] + cat_vars

# ===========================
# 3. Create grouping key
# ===========================
data["group_key"] = data[group_cols].astype(str).agg("_".join, axis=1)

# ===========================
# 4. Build clusters
# ===========================
groups = data.groupby("group_key")
clusters = [g.index.tolist() for _, g in groups if len(g) > 1]

print("Nombre de groupes similaires (round-to-int) :", len(clusters))

def analyze_target_round_int(target):
    same = 0
    diff = 0

    for grp in clusters:
        votes = list(data.loc[grp, target])
        if len(set(votes)) == 1:
            same += 1
        else:
            diff += 1

    summary = {
        "target": target,
        "total_groups": len(clusters),
        "identical_groups": same,
        "different_groups": diff,
        "agreement_rate": round(same / len(clusters), 3),
        "disagreement_rate": round(diff / len(clusters), 3),
    }

    print(f"\n=== Analyse pour {target} ===")
    for k, v in summary.items():
        print(f"{k} : {v}")

    return summary

def get_example_groups_round_int(target, N_EXAMPLES=10):
    examples = []

    for grp in clusters:
        votes = list(data.loc[grp, target])
        if len(set(votes)) > 1:
            examples.append({
                "indices": grp,
                "votes": votes,
                "size": len(grp),
                "context": data.loc[grp, group_cols].drop_duplicates()
            })
        if len(examples) >= N_EXAMPLES:
            break

    return examples

def save_results_round_int(target, summary, examples):
    out_dir = OUTPUT_DIR / target
    out_dir.mkdir(exist_ok=True)

    with open(out_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=4)

    detailed_list = []

    for ex in examples:
        detailed_list.append({
            "size": ex["size"],
            "votes": ex["votes"],
            "context": ex["context"].to_dict(orient="records")
        })

    with open(out_dir / "contradictory_examples.json", "w") as f:
        json.dump(detailed_list, f, indent=4)

    print(f"→ Saved results to {out_dir}")

global_summary = {}

for target in targets:
    print("\n" + "="*80)
    print(f"🔍 ANALYSE ROUND-TO-INT — {target.upper()}")
    print("="*80)

    summary = analyze_target_round_int(target)
    global_summary[target] = summary

    examples = get_example_groups_round_int(target, N_EXAMPLES=10)

    print(f"\nTop {len(examples)} contradictory groups for {target}:")
    for i, ex in enumerate(examples, start=1):
        print(f"\n--- Groupe #{i} ---")
        print("Taille :", ex["size"])
        print("Votes :", ex["votes"])
        print("\nContexte (arrondi + catégories) :")
        print(ex["context"])

    save_results_round_int(target, summary, examples)

with open(OUTPUT_DIR / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=4)


Dataset size: 15224
Nombre de groupes similaires (round-to-int) : 1029

🔍 ANALYSE ROUND-TO-INT — THERMAL_SENSATION

=== Analyse pour thermal_sensation ===
target : thermal_sensation
total_groups : 1029
identical_groups : 464
different_groups : 565
agreement_rate : 0.451
disagreement_rate : 0.549

Top 10 contradictory groups for thermal_sensation:

--- Groupe #1 ---
Taille : 2
Votes : [-2, 0]

Contexte (arrondi + catégories) :
       Tair_r  RH_r  clo_r  vel_r  Tout_r  Met_r   Âge  Sexe  Season Climate  \
10653      10    52      2      0      12      1  53.0  male  winter     Cfa   

      Building_type          cooling type  
10653        office  naturally ventilated  

--- Groupe #2 ---
Taille : 2
Votes : [-1, -2]

Contexte (arrondi + catégories) :
       Tair_r  RH_r  clo_r  vel_r  Tout_r  Met_r   Âge    Sexe  Season  \
10830      10    56      2      0      12      1  20.0  female  winter   

      Climate Building_type          cooling type  
10830     Cfa        office  naturally

Tolérance ±

"""
Case B — Tolerance Window Analysis (± Range per Variable)
=========================================================

This analysis introduces a realistic tolerance window around each environmental
and personal variable:

- Tair, Tout: ±1°C  
- RH: ±5%  
- vel: ±0.3 m/s  
- clo: ±0.3  
- Met: ±0.2  

Categorical variables must match strictly.

Objective
---------

Identify pairs of occupants experiencing *practically identical* conditions and
measure the consistency of their thermal comfort votes.

Methodology
-----------

1. Apply strict matching on all categorical variables.
2. Apply tolerance windows on numerical variables using `.between()`.
3. Generate all (i, j) pairs that satisfy the tolerance constraints.
4. Compute agreement and disagreement rates for each target variable.
5. Export JSON files containing all contradictory pairs.

"""


In [4]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# ===========================
# 1. Load dataset
# ===========================
df = pd.read_csv("../Data/ASHRAE_2022_Clean_api.csv")

features_num = ['Tair', 'RH', 'clo', 'vel', 'Âge', 'Tout', 'Met']
features_cat = ['Sexe', 'Season', 'Climate', 'Building_type', 'cooling type']

targets = ['thermal_sensation', 'TSV_3p', 'thermal_preference']
targets = [t for t in targets if t in df.columns]

data = df[features_num + features_cat + targets].dropna().reset_index(drop=True)

print("Dataset utilisé :", len(data), "lignes")

OUTPUT_DIR = Path("Votes_analysis/analysis_results_tolerance_context")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tol = {
    'Tair': 1,
    'Tout': 1,
    'RH': 5,
    'vel': 0.3,
    'clo': 0.3,
    'Met': 0.2
}

pairs = []

for i in range(len(data)):
    ref = data.iloc[i]

    mask_cat = (
        (data['Sexe'] == ref['Sexe']) &
        (data['Season'] == ref['Season']) &
        (data['Climate'] == ref['Climate']) &
        (data['Building_type'] == ref['Building_type']) &
        (data['cooling type'] == ref['cooling type']) &
        (data['Âge'] == ref['Âge'])
    )

    subset = data[mask_cat]

    mask_num = (
        (subset['Tair'].between(ref['Tair'] - tol['Tair'], ref['Tair'] + tol['Tair'])) &
        (subset['Tout'].between(ref['Tout'] - tol['Tout'], ref['Tout'] + tol['Tout'])) &
        (subset['RH'].between(ref['RH'] - tol['RH'], ref['RH'] + tol['RH'])) &
        (subset['vel'].between(ref['vel'] - tol['vel'], ref['vel'] + tol['vel'])) &
        (subset['clo'].between(ref['clo'] - tol['clo'], ref['clo'] + tol['clo'])) &
        (subset['Met'].between(ref['Met'] - tol['Met'], ref['Met'] + tol['Met']))
    )

    final = subset[mask_num]

    for j in final.index:
        if j > i:
            pairs.append((i, j))

print("Nombre total de paires similaires trouvées :", len(pairs))

def analyze_pairs(target):
    same = 0
    diff = 0

    mismatches = []

    for i, j in pairs:
        v1 = data.loc[i, target]
        v2 = data.loc[j, target]

        if v1 == v2:
            same += 1
        else:
            diff += 1
            mismatches.append({
                "i": int(i),
                "j": int(j),
                "vote_i": int(v1),
                "vote_j": int(v2),
                "context_i": data.loc[i, features_num + features_cat].to_dict(),
                "context_j": data.loc[j, features_num + features_cat].to_dict()
            })

    summary = {
        "target": target,
        "num_pairs": len(pairs),
        "identical_votes": same,
        "different_votes": diff,
        "agreement_rate": round(same / len(pairs), 3),
        "disagreement_rate": round(diff / len(pairs), 3)
    }

    print(f"\n=== Analyse {target} ===")
    for k, v in summary.items():
        print(f"{k} : {v}")

    return summary, mismatches

global_summary = {}

for target in targets:
    print("\n" + "="*70)
    print(f"🔍 ANALYSE (TOLÉRANCE ±) — {target.upper()}")
    print("="*70)

    summary, mismatches = analyze_pairs(target)
    global_summary[target] = summary

    out_dir = OUTPUT_DIR / target
    out_dir.mkdir(exist_ok=True)

    with open(out_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=4)

    with open(out_dir / "mismatches.json", "w") as f:
        json.dump(mismatches, f, indent=4)

    print(f"→ Résultats enregistrés dans {out_dir}")

with open(OUTPUT_DIR / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=4)

print("\n🔥 Analyse terminée (tolérances ±).")


Dataset utilisé : 15224 lignes
Nombre total de paires similaires trouvées : 21410

🔍 ANALYSE (TOLÉRANCE ±) — THERMAL_SENSATION

=== Analyse thermal_sensation ===
target : thermal_sensation
num_pairs : 21410
identical_votes : 9470
different_votes : 11940
agreement_rate : 0.442
disagreement_rate : 0.558
→ Résultats enregistrés dans Votes_analysis/analysis_results_tolerance_context/thermal_sensation

🔍 ANALYSE (TOLÉRANCE ±) — THERMAL_PREFERENCE

=== Analyse thermal_preference ===
target : thermal_preference
num_pairs : 21410
identical_votes : 12853
different_votes : 8557
agreement_rate : 0.6
disagreement_rate : 0.4
→ Résultats enregistrés dans Votes_analysis/analysis_results_tolerance_context/thermal_preference

🔥 Analyse terminée (tolérances ±).


KNN analysis

"""
Case C — Nearest Neighbor Analysis (KNN in Normalized Feature Space)
====================================================================

In this analysis, each occupant's conditions are represented as a
multidimensional feature vector combining:

- normalized numerical variables (Z-score),
- encoded categorical variables (integer codes).

For each individual, we then identify **the closest neighbor** in terms of
Euclidean distance within this full feature space.

Objective
---------

Evaluate the agreement between the thermal comfort votes of individuals who
experience a context that is truly similar in the complete multidimensional
space, without relying on arbitrary rounding or tolerance thresholds.

Methodology
-----------

1. Apply Z-score normalization to all numerical variables.
2. Encode categorical variables as integer labels.
3. Use `NearestNeighbors(n_neighbors=2)`:
   - index 0 → the individual itself,
   - index 1 → its closest neighbor.
4. Compare the vote of each individual with that of their nearest neighbor.
5. Identify and extract the closest contradictory pairs.
6. Export results as JSON and CSV files.

"""


In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import json
from pathlib import Path

df = pd.read_csv("../Data/ASHRAE_2022_Clean_api.csv")

num_feats = ['Tair', 'RH', 'clo', 'vel', 'Âge', 'Tout', 'Met']
cat_feats = ['Sexe', 'Season', 'Climate', 'Building_type', 'cooling type']

features = num_feats + cat_feats

targets = ['thermal_sensation', 'TSV_3p', 'thermal_preference']
targets = [t for t in targets if t in df.columns]

data = df[features + targets].dropna().reset_index(drop=True)

print("Dataset utilisé :", len(data), "lignes")

OUTPUT_DIR = Path("Votes_analysis/analysis_results_knn_neighbor")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_enc = data[features].copy()

df_enc[cat_feats] = df_enc[cat_feats].astype("category").apply(lambda x: x.cat.codes)

scaler = StandardScaler()
df_enc[num_feats] = scaler.fit_transform(df_enc[num_feats])

knn = NearestNeighbors(n_neighbors=2, metric='euclidean')
knn.fit(df_enc)

distances, indices = knn.kneighbors(df_enc)

nn_index = indices[:, 1]
nn_distance = distances[:, 1]

def compare_target(target):
    y = data[target].values
    nn_y = y[nn_index]
    same = (y == nn_y)

    agreement = same.mean()
    disagreement = 1 - agreement

    df_res = pd.DataFrame({
        "self_vote": y,
        "nn_vote": nn_y,
        "same": same,
        "distance": nn_distance,
        "self_idx": data.index,
        "nn_idx": nn_index
    })

    return df_res, agreement, disagreement
global_summary = {}

for target in targets:
    print("\n" + "="*80)
    print(f"🔍 ANALYSE KNN — {target.upper()}")
    print("="*80)

    df_res, acc, err = compare_target(target)

    print(f"Agreement: {round(acc,3)} | Disagreement: {round(err,3)}")

    out_dir = OUTPUT_DIR / target
    out_dir.mkdir(exist_ok=True)

    df_res.to_csv(out_dir / "knn_results.csv", index=False)

    summary = {
        "target": target,
        "agreement_rate": float(acc),
        "disagreement_rate": float(err),
        "num_samples": len(df_res)
    }
    global_summary[target] = summary

    with open(out_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=4)

with open(OUTPUT_DIR / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=4)


TOP_K = 20

for target in targets:
    print("\n" + "="*80)
    print(f"🔥 Top {TOP_K} contradictions les plus proches — {target}")
    print("="*80)

    df_res, _, _ = compare_target(target)

    contr = df_res[df_res.same == False].sort_values("distance").head(TOP_K)
    display(contr)

    out_dir = OUTPUT_DIR / target
    contr.to_csv(out_dir / "closest_contradictions.csv", index=False)


Dataset utilisé : 15224 lignes

🔍 ANALYSE KNN — THERMAL_SENSATION
Agreement: 0.431 | Disagreement: 0.569

🔍 ANALYSE KNN — THERMAL_PREFERENCE
Agreement: 0.593 | Disagreement: 0.407

🔥 Top 20 contradictions les plus proches — thermal_sensation


,self_vote,nn_vote,same,distance,self_idx,nn_idx
3377,1,0,False,0.000000,3377,3376
11041,-2,-3,False,0.000000,11041,11042
10737,-1,-2,False,0.000000,10737,10734
1231,0,2,False,0.000000,1231,1229
1394,1,-1,False,0.000000,1394,1392
13686,-1,0,False,0.000000,13686,13685
1382,3,2,False,0.000000,1382,1384
13976,2,0,False,0.000000,13976,13974
10181,2,1,False,0.020896,10181,10180
10180,1,2,False,0.020896,10180,10181



🔥 Top 20 contradictions les plus proches — thermal_preference


,self_vote,nn_vote,same,distance,self_idx,nn_idx
1674,1,-1,False,0.000000,1674,1672
13686,1,0,False,0.000000,13686,13685
1662,0,-1,False,0.000000,1662,1672
9029,0,1,False,0.000000,9029,9027
13655,0,-1,False,0.000000,13655,13661
2831,0,1,False,0.000000,2831,2841
10337,-1,0,False,0.018416,10337,10338
10338,0,-1,False,0.018416,10338,10337
10008,0,-1,False,0.023394,10008,9954
9954,-1,0,False,0.023394,9954,10008
